# Lab 01: Spark Architecture & Cluster Exploration

## Objectives
- Understand the components of a Spark cluster (driver, executors, cluster manager)
- Explore SparkSession and SparkContext configuration
- Navigate the Spark UI to inspect jobs, stages, and tasks
- Examine how Spark partitions data and distributes work

## Exam Domain
- **Apache Spark Architecture and Components — 20%**

## Estimates
- **Time:** ~25 minutes
- **Cost:** $1-2 (requires cluster)
- **Compute:** Single-node cluster (see prerequisites.md)

![Cluster Architecture](../assets/diagrams/lab-01-cluster-architecture.png)

In [ ]:
# This lab requires a cluster (not serverless) to explore the Spark UI
# Verify setup-catalog.py has been run before starting

USE_CATALOG = "spark_lab_guide"
USE_SCHEMA = "default"

spark.sql(f"USE CATALOG {USE_CATALOG}")
spark.sql(f"USE SCHEMA {USE_SCHEMA}")

print(f"Spark version: {spark.version}")
print(f"Catalog: {USE_CATALOG}")

## A. SparkSession — Your Entry Point

Every Spark application starts with a `SparkSession`. In Databricks, it is pre-created as `spark`. The SparkSession provides access to the DataFrame API, SQL engine, and cluster configuration.

In [ ]:
# SparkSession is already available as 'spark' in Databricks
# Inspect its configuration — some configs vary between cluster and serverless
def safe_conf_get(key, default="N/A (serverless)"):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

print(f"App name: {safe_conf_get('spark.app.name')}")
print(f"Master: {safe_conf_get('spark.master')}")
print(f"Default parallelism: {safe_conf_get('spark.default.parallelism')}")
print(f"Shuffle partitions: {safe_conf_get('spark.sql.shuffle.partitions')}")
print(f"AQE enabled: {safe_conf_get('spark.sql.adaptive.enabled')}")

## B. Exploring Cluster Resources

The SparkContext (accessible via `spark.sparkContext`) connects to the cluster manager and manages executors. Let's inspect what resources are available.

In [ ]:
# SparkContext access — available on clusters, limited on serverless
try:
    sc = spark.sparkContext
    print(f"SparkContext version: {sc.version}")
    print(f"Application ID: {sc.applicationId}")
    print(f"Default parallelism: {sc.defaultParallelism}")
    print(f"Default min partitions: {sc.defaultMinPartitions}")
    print(f"SparkContext UI: {sc.uiWebUrl}")
except Exception as e:
    print(f"SparkContext not directly accessible (serverless/Spark Connect mode)")
    print(f"Spark version: {spark.version}")
    print("Note: On serverless, use the Databricks UI to access query plans and metrics.")

## C. Jobs, Stages, and Tasks

When you call an **action** (like `count()` or `show()`), Spark creates a **job**. Each job is divided into **stages** (separated by shuffle boundaries), and each stage contains **tasks** (one per partition).

Let's trigger a job and observe the hierarchy.

In [ ]:
# Read a sample of taxi data to trigger a job
taxi_path = "/Volumes/spark_lab_guide/default/nyc_taxi"
df = spark.read.parquet(taxi_path)

# This action triggers a job — check the Spark UI Jobs tab
row_count = df.count()
print(f"Total taxi trips: {row_count:,}")

In [ ]:
# A more complex query creates multiple stages
# groupBy triggers a shuffle, creating a stage boundary
from pyspark.sql.functions import col, count, avg

result = (
    df.groupBy("PULocationID")
    .agg(
        count("*").alias("trip_count"),
        avg("total_amount").alias("avg_total"),
    )
    .orderBy(col("trip_count").desc())
    .limit(10)
)

# This triggers a multi-stage job — inspect the Spark UI SQL tab
result.show()

> **Exam Tip:** The exam tests your understanding of the relationship between jobs, stages, and tasks. Remember:
> - **1 action = 1 job**
> - **1 shuffle = 1 stage boundary**
> - **1 partition = 1 task per stage**

## D. Partitions and Parallelism

Spark divides data into **partitions** — each partition is processed by one task on one core. The number of partitions determines the level of parallelism.

In [ ]:
# Check how many partitions the taxi data has
try:
    print(f"Number of partitions: {df.rdd.getNumPartitions()}")
except Exception:
    print("Partition count not available on serverless (RDD API limited)")

# Repartition to control parallelism
df_4 = df.repartition(4)
try:
    print(f"After repartition(4): {df_4.rdd.getNumPartitions()}")
except Exception:
    print("repartition(4) applied (partition count not queryable on serverless)")

# Coalesce reduces partitions WITHOUT a full shuffle
df_2 = df_4.coalesce(2)
try:
    print(f"After coalesce(2): {df_2.rdd.getNumPartitions()}")
except Exception:
    print("coalesce(2) applied (partition count not queryable on serverless)")

In [ ]:
from pyspark.sql.functions import spark_partition_id

# Repartition by a column — useful for join/aggregation optimization
df_by_location = df.repartition(8, "PULocationID")
try:
    print(f"Partitions by PULocationID: {df_by_location.rdd.getNumPartitions()}")
except Exception:
    print("repartition(8, 'PULocationID') applied")

# Check partition sizes using spark_partition_id()
partition_counts = (
    df_by_location.withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)
partition_counts.show()

## E. Navigating the Spark UI

Open the **Spark UI** link printed in Section B (or use the Databricks cluster UI). Key tabs to explore:

| Tab | What to Look For |
|-----|-----------------|
| **Jobs** | Number of jobs triggered, their status, and duration |
| **Stages** | Shuffle read/write bytes, number of tasks per stage |
| **Storage** | Cached DataFrames (none yet — we'll cover caching in Lab 10) |
| **Environment** | All Spark configuration properties |
| **Executors** | Memory usage, active tasks, shuffle metrics |
| **SQL/DataFrame** | Visual query plans for DataFrame operations |

Run the cell below and then check the SQL tab to see the query plan.

In [ ]:
# Trigger a query and inspect it in Spark UI > SQL tab
detailed_query = (
    df.filter(col("trip_distance") > 5)
    .filter(col("total_amount") > 20)
    .groupBy("payment_type")
    .agg(
        count("*").alias("trips"),
        avg("tip_amount").alias("avg_tip"),
    )
)

# explain() shows the plan without executing
print("=== Physical Plan ===")
detailed_query.explain()

print("\n=== Extended Plan (logical + physical) ===")
detailed_query.explain(True)

## Exam-Style Review

**Q1.** What is the role of the Driver in a Spark application?
- A) It stores the data partitions
- B) It runs the main program, creates the SparkSession, and coordinates task execution
- C) It manages cluster resources and assigns workers
- D) It only executes tasks assigned by the cluster manager

**Q2.** What determines the number of tasks in a stage?
- A) The number of executors
- B) The number of cores
- C) The number of partitions
- D) The value of spark.sql.shuffle.partitions

**Q3.** What is the difference between `repartition()` and `coalesce()`?
- A) They are identical
- B) `repartition()` can increase or decrease partitions (full shuffle); `coalesce()` can only decrease (no full shuffle)
- C) `coalesce()` is always faster
- D) `repartition()` does not trigger a shuffle

**Q4.** Which Spark UI tab shows the visual query plan for DataFrame operations?
- A) Jobs
- B) Stages
- C) SQL/DataFrame
- D) Environment

### Answers
- **Q1: B** — The Driver runs the main program, creates SparkSession, and coordinates work across executors.
- **Q2: C** — Each partition gets one task per stage. The number of partitions determines task count.
- **Q3: B** — `repartition()` does a full shuffle and can increase or decrease partitions. `coalesce()` avoids a full shuffle but can only decrease partitions.
- **Q4: C** — The SQL/DataFrame tab shows visual query plans for DataFrame and SQL operations.

## Key Takeaways
- SparkSession is the entry point; SparkContext manages the cluster connection
- Jobs are triggered by actions, divided into stages at shuffle boundaries, with one task per partition
- `repartition()` triggers a full shuffle; `coalesce()` avoids it but can only reduce partitions
- The Spark UI is essential for understanding query execution and diagnosing performance

**Next:** [Lab 02 — Reading & Writing Data Formats](02-reading-writing-data-formats.ipynb)

In [ ]:
# Cleanup — drop temp views and uncache
# (No temp views or cached data in this lab)
print("Lab 01 complete. No cleanup needed.")